# Mini Search Engine Berita IHSG (CNBC Indonesia)
Final Project Information Retrieval

## Langkah 1: Instalasi
Tujuan: Menginstal dependensi pustaka yang dibutuhkan (`requirements.txt`).

In [ ]:
%pip install -r requirements.txt

## Langkah 2: Konfigurasi dan Impor
Tujuan: Mengimpor pustaka dan mendefinisikan konstanta/parameter terpusat (seed, path, batasan crawling, model dense, bobot RRF).

In [ ]:
import hashlib
import json
import math
import os
import pickle
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# Menetapkan seed acak demi reprodusibilitas eksperimen
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Konfigurasi crawler
CONTACT_EMAIL = "dickyferdy5@gmail.com"
TARGET_DOCS = 2000
CRAWL_DELAY = 2.0
MAX_FETCHES = 8000

# Konfigurasi retriever leksikal (BM25 Okapi)
BM25_K1 = 1.5
BM25_B = 0.75

# Konfigurasi retriever dense (multilingual-e5-small)
EMBED_MODEL = "intfloat/multilingual-e5-small"
EMBED_MAX_LEN = 256
EMBED_BATCH = 64

# Konfigurasi hybrid fusion & kedalaman evaluasi
RRF_K = 60
CANDIDATES = 100
POOL_DEPTH = 10
EVAL_K = 10

# Jalur direktori dan berkas (relatif terhadap direktori kerja notebook)
PATH_DATA_DIR = Path("data")
PATH_INDEXES_DIR = Path("indexes")
PATH_RESULTS_DIR = Path("results")

PATH_QUERIES = PATH_DATA_DIR / "queries.csv"
PATH_CORPUS = PATH_DATA_DIR / "corpus.jsonl"
PATH_CORPUS_URLS = PATH_DATA_DIR / "corpus_urls.txt"
PATH_POOL_TO_LABEL = PATH_DATA_DIR / "pool_to_label.csv"
PATH_QRELS = PATH_DATA_DIR / "qrels.csv"
PATH_VISITED_URLS = PATH_DATA_DIR / "visited_urls.txt"
PATH_KNOWN_URLS = PATH_DATA_DIR / "known_urls.txt"

PATH_BM25_TOKENS = PATH_INDEXES_DIR / "bm25_tokens.pkl"
PATH_FAISS_INDEX = PATH_INDEXES_DIR / "faiss.index"
PATH_DENSE_IDS = PATH_INDEXES_DIR / "dense_ids.json"

PATH_PER_QUERY = PATH_RESULTS_DIR / "per_query.csv"
PATH_SUMMARY = PATH_RESULTS_DIR / "summary.csv"
PATH_SIGNIFICANCE = PATH_RESULTS_DIR / "significance.csv"
PATH_ERROR_ANALYSIS = PATH_RESULTS_DIR / "error_analysis.csv"

# Memastikan seluruh direktori output tersedia
for path_dir in [PATH_DATA_DIR, PATH_INDEXES_DIR, PATH_RESULTS_DIR]:
    path_dir.mkdir(parents=True, exist_ok=True)

print("Konfigurasi dan pustaka berhasil dimuat. Direktori siap.")

## Langkah 3: Ambil Korpus (Crawler)
Tujuan: Mengumpulkan 2.000 artikel berita pasar modal/IHSG dari CNBC Indonesia secara terfokus, etis, dan idempoten.

## Langkah 4: Praproses, Retriever, Bangun Indeks
Tujuan: Melakukan pembersihan teks (case folding, tokenisasi, normalisasi, stopword removal, stemming terproteksi) serta membangun indeks BM25 dan indeks dense FAISS.

## Langkah 5: Muat Sistem, Ukur Memori, Fungsi Cari
Tujuan: Memuat model dan indeks, mengestimasi footprint RAM/RSS proses, serta menyiapkan fungsi antarmuka seragam `search(query, k)` untuk BM25, Dense, dan Hybrid RRF.

## Langkah 6: Pooling
Tujuan: Mengambil top-10 kandidat dokumen dari BM25, Dense, dan Hybrid untuk setiap kueri uji, menggabungkan secara unik, dan mengacaknya ke `data/pool_to_label.csv`.

## Langkah 7: Pelabelan Manual dan Pembaca Qrels
Tujuan: Membaca hasil relevance judgement manual manusia secara andal (`read_csv_robust`) dan menghasilkan ground truth `data/qrels.csv`.

## Langkah 8: Evaluasi Efektivitas
Tujuan: Menghitung metrik performa retrieval (nDCG@10, MAP, MRR, P@10) per kueri dan rata-rata sistem untuk membandingkan BM25 vs Dense vs Hybrid RRF.

## Langkah 9: Evaluasi Efisiensi
Tujuan: Mengukur efisiensi sistem berupa latensi pencarian (dengan fase pemanasan dan rotasi bergilir), konsumsi memori RSS, serta ukuran berkas indeks di disk.

## Langkah 10: Signifikansi dan Analisis Error
Tujuan: Melakukan uji signifikansi statistik (paired t-test dan Wilcoxon) antarmetode serta memetakan kueri terbaik dan terburuk pada sistem hybrid.

## Langkah 11: Antarmuka
Tujuan: Menyediakan widget pencarian interaktif di dalam notebook (`ipywidgets`) dengan opsi pemilihan metode dan visualisasi cuplikan hasil.